In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from factor_analyzer import FactorAnalyzer
from sklearn.linear_model import Lasso
from transformers import pipeline
from transformers import AutoTokenizer, AutoModel
import torch

traits = {
    'cOPN': 'Openness',
    'cEXT': 'Extraversion',
    'cNEU': 'Neuroticism',
    'cAGR': 'Agreeableness',	
    'cCON': 'Conscientiousness'
}


In [6]:
# # Load the model and tokenizer
# model_name = "deepseekai/deepseek-base"  # Hypothetical model name
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)

# # Function to generate embeddings
# def generate_embedding(text):
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
#     with torch.no_grad():
#         outputs = model(**inputs)
#     embedding = outputs.last_hidden_state[:, 0, :].squeeze()
#     return embedding

# # Example usage
# text = "This is a sample text to generate embeddings."
# embedding = generate_embedding(text)
# print(embedding)

In [12]:
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForCausalLM

config = AutoConfig.from_pretrained("deepseek-ai/DeepSeek-R1", trust_remote_code=True)
del config.quantization_config
config


DeepseekV3Config {
  "_name_or_path": "deepseek-ai/DeepSeek-R1",
  "architectures": [
    "DeepseekV3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "deepseek-ai/DeepSeek-R1--configuration_deepseek.DeepseekV3Config",
    "AutoModel": "deepseek-ai/DeepSeek-R1--modeling_deepseek.DeepseekV3Model",
    "AutoModelForCausalLM": "deepseek-ai/DeepSeek-R1--modeling_deepseek.DeepseekV3ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "eos_token_id": 1,
  "ep_size": 1,
  "first_k_dense_replace": 3,
  "hidden_act": "silu",
  "hidden_size": 7168,
  "initializer_range": 0.02,
  "intermediate_size": 18432,
  "kv_lora_rank": 512,
  "max_position_embeddings": 163840,
  "model_type": "deepseek_v3",
  "moe_intermediate_size": 2048,
  "moe_layer_freq": 1,
  "n_group": 8,
  "n_routed_experts": 256,
  "n_shared_experts": 1,
  "norm_topk_prob": true,
  "num_attention_heads": 128,
  "num_experts_per_tok": 8,
  "num_hidden_layers": 61

In [13]:
# config
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1", config=config, trust_remote_code=True)
# Use a pipeline as a high-level helper

model-00008-of-000163.safetensors:  65%|######5   | 2.81G/4.30G [00:00<?, ?B/s]

model-00009-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

model-00010-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

model-00011-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

model-00012-of-000163.safetensors:   0%|          | 0.00/1.32G [00:00<?, ?B/s]

...

model-00016-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

model-00017-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

/home/jmaharja/anaconda3/envs/gpu/lib/python3.8/site-packages/huggingface_hub/file_download.py:651: UserWarning: Not enough free disk space to download the file. The expected file size is: 4302.35 MB. The target location /home/jmaharja/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1/blobs only has 4101.23 MB free disk space.
  warnings.warn(


model-00018-of-000163.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
from transformers import AutoModelForCausalLM, AutoConfig
cache_dir  ="/data3/jm/hf_home/"
config = AutoConfig.from_pretrained("deepseek-ai/DeepSeek-R1",  cache_dir = cache_dir, local_files_only=True)
del config.quantization_config

config
import pandas as pd
import numpy as np
from collections import defaultdict
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import torch
from transformers import AutoModelForCausalLM, AutoConfig

def process_embeddings(df, model_name, batch_size=8):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f'Generating Embedding from {model_name} using {device}')
    cache_dir  ="/data3/jm/hf_home/"
    config = AutoConfig.from_pretrained(model_name, cache_dir = cache_dir, local_files_only=True)
    del config.quantization_config
    model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1", cache_dir = cache_dir, local_files_only=True)
    model.to(device)
    model.eval()  
    embeddings_list = []
    
    for i in range(0, len(df), batch_size):
        batch_texts = df['STATUS'][i:i + batch_size].tolist()
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings_list.append(cls_embeddings.cpu().numpy())
    print(f'Embedding Completed for {model_name}')
    return np.vstack(embeddings_list)

s = df.sample(100)
re = process_embeddings(s['STATUS'], 'deepseek-ai/DeepSeek-V3-Base')
# re
